In [ ]:
#db/init.sql
CREATE EXTENSION IF NOT EXISTS postgis;

CREATE TABLE IF NOT EXISTS hazard_reports (
  id         BIGSERIAL PRIMARY KEY,
  type       TEXT NOT NULL,                       -- '계단','점자블록부재','공사' 등
  severity   INTEGER NOT NULL CHECK (severity BETWEEN 1 AND 5),
  note       TEXT,
  created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
  geom       geometry(Point, 4326) NOT NULL       -- (lng, lat)
);

CREATE INDEX IF NOT EXISTS idx_hazard_reports_geom
  ON hazard_reports
  USING GIST (geom);


In [ ]:
#api/main.py
from fastapi import FastAPI, Query
from pydantic import BaseModel, Field
from typing import List, Literal
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()
DATABASE_URL = os.getenv(
    "DATABASE_URL",
    "postgresql+psycopg://postgres:postgres@db:5432/safe_map",
)

engine = create_engine(DATABASE_URL, future=True)
app = FastAPI(title="Safe Route API", version="0.1.0")

# ---------- Pydantic Schemas ----------
class ReportIn(BaseModel):
    type: str = Field(..., description="예: 계단, 점자블록부재, 공사")
    severity: int = Field(..., ge=1, le=5)
    location: List[float] = Field(..., min_items=2, max_items=2, description="[lng, lat]")
    note: str | None = None

def feature(row) -> dict:
    # row: (id, type, severity, note, created_at, lng, lat)
    return {
        "type": "Feature",
        "id": row[0],
        "properties": {
            "type": row[1],
            "severity": row[2],
            "note": row[3],
            "created_at": row[4].isoformat() if row[4] else None,
        },
        "geometry": {"type": "Point", "coordinates": [row[5], row[6]]},
    }

# ---------- Endpoints ----------

@app.post("/reports")
def create_report(rep: ReportIn):
    lng, lat = rep.location
    sql = text("""
        INSERT INTO hazard_reports (type, severity, note, geom)
        VALUES (:type, :severity, :note, ST_SetSRID(ST_Point(:lng, :lat), 4326))
        RETURNING id
    """)
    with engine.begin() as conn:
        new_id = conn.execute(sql, dict(type=rep.type, severity=rep.severity,
                                        note=rep.note, lng=lng, lat=lat)).scalar_one()
    return {"ok": True, "id": new_id}

@app.get("/reports")
def list_reports(
    bbox: str = Query(..., description="minLng,minLat,maxLng,maxLat"),
    types: str | None = Query(None, description="콤마구분: 계단,점자블록부재"),
):
    # bbox 파싱
    try:
        min_lng, min_lat, max_lng, max_lat = [float(x) for x in bbox.split(",")]
    except Exception:
        return {"features": [], "error": "bbox format: minLng,minLat,maxLng,maxLat"}

    type_filter_sql = ""
    params = dict(min_lng=min_lng, min_lat=min_lat, max_lng=max_lng, max_lat=max_lat)
    if types:
        typelist = [t.strip() for t in types.split(",") if t.strip()]
        # ANY(:types) 사용이 편하지만 SQLAlchemy 바인딩 단순화를 위해 IN (...) 구성
        in_params = {f"t{i}": v for i, v in enumerate(typelist)}
        type_filter_sql = "AND type IN (" + ", ".join(f":{k}" for k in in_params) + ")"
        params.update(in_params)

    sql = text(f"""
        SELECT id, type, severity, note, created_at,
               ST_X(geom) AS lng, ST_Y(geom) AS lat
        FROM hazard_reports
        WHERE geom && ST_MakeEnvelope(:min_lng, :min_lat, :max_lng, :max_lat, 4326)
          AND ST_Intersects(
                geom,
                ST_MakeEnvelope(:min_lng, :min_lat, :max_lng, :max_lat, 4326)
              )
          {type_filter_sql}
        ORDER BY created_at DESC
        LIMIT 1000
    """)
    with engine.begin() as conn:
        rows = conn.execute(sql, params).all()
    return {"type": "FeatureCollection", "features": [feature(r) for r in rows]}

@app.get("/route")
def route(
    origin: str,
    dest: str,
    pref: Literal["safe", "balanced", "fast"] = "safe",
):
    """
    MVP: 외부 라우터 연결 전이므로 목업 후보 2~3개에 간단 가중치 적용.
    추후 Valhalla/카카오 경로 후보 + 위험점수를 합산하도록 교체.
    """
    cand = [
        {"id": "A", "distance_m": 1200, "duration_s": 1000, "risk_score": 2.7,
         "coords": [[127.3, 37.0], [127.31, 37.01]]},
        {"id": "B", "distance_m": 1100, "duration_s": 950, "risk_score": 3.2,
         "coords": [[127.3, 37.0], [127.305, 37.012]]},
    ]
    if pref == "fast":
        chosen = min(cand, key=lambda x: x["duration_s"])
    elif pref == "balanced":
        chosen = min(cand, key=lambda x: 0.5*x["duration_s"] + 0.5*1000*x["risk_score"])
    else:  # safe
        chosen = min(cand, key=lambda x: x["risk_score"])

    return {
        "route_id": chosen["id"],
        "preference": pref,
        "distance_m": chosen["distance_m"],
        "duration_s": chosen["duration_s"],
        "risk_score": chosen["risk_score"],
        "polyline": chosen["coords"],     # 실제로는 encoded polyline 권장
        "steps": [{"instruction": "직진 100m", "distance_m": 100}],
        "alternatives": [
            {k: c[k] for k in ("id", "distance_m", "duration_s", "risk_score")}
            for c in cand if c["id"] != chosen["id"]
        ],
    }


실행
cp .env.example .env
docker compose up --build